### Online Retail II — Customer-Month Feature Table & Churn Labeling

Takes the cleaned dataset (`data/online_retail_ii_cleaned.parquet`) and prepares it for Exploratory Data Analysis (EDA) (Task #5):

1. Load + inspect standardized data
2. Aggregate customer-month features: purchase frequency, total spend, average order value, unique items purchases
3. Compute days-since-last-purchase using only prior data
4. Apply the forward-looking 90-day churn label
5. Exclude customer-months without a full, observable 90-day window
6. Save the customer-month feature table

## 1. Load cleaned data

In [ ]:
# imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/online_retail_ii_cleaned.parquet')
print(df.shape)
df[['Invoice', 'Customer ID', 'InvoiceDate', 'Quantity', 'Price', 'Total_Price', 'StockCode', 'CustomerMonth']].head()


(608568, 10)


,Invoice,Customer ID,InvoiceDate,Quantity,Price,Total_Price,StockCode,CustomerMonth
0,489434,13085,2009-12-01 07:45:00,12,6.95,83.4,85048,2009-12
1,489434,13085,2009-12-01 07:45:00,12,6.75,81.0,79323P,2009-12
2,489434,13085,2009-12-01 07:45:00,12,6.75,81.0,79323W,2009-12
3,489434,13085,2009-12-01 07:45:00,48,2.10,100.8,22041,2009-12
4,489434,13085,2009-12-01 07:45:00,24,1.25,30.0,21232,2009-12


## 2. Aggregate customer-month features

For each customer-month, only transactions within that month are used to compute purchase frequency, total spend, average order value, and unique items

In [11]:
monthly = (
    df.groupby(['Customer ID', 'CustomerMonth'])
    .agg(
        purchase_frequency = ('Invoice', 'nunique'),
        total_spend = ('Total_Price', 'sum'),
        unique_items = ('StockCode', 'nunique'),
        first_purchase_date = ('InvoiceDate', 'min'),
        last_purchase_date = ('InvoiceDate', 'max')
    )
    .reset_index()
    .sort_values(['Customer ID', 'CustomerMonth'])
    .reset_index(drop=True)
)

monthly['avg_order_value'] = monthly['total_spend'] / monthly['purchase_frequency']

print(f"Customer-month rows: {len(monthly):,}")
print(f"Unique customers: {monthly['Customer ID'].nunique():,}")
monthly.head()

Customer-month rows: 20,609
Unique customers: 5,224


,Customer ID,CustomerMonth,purchase_frequency,total_spend,unique_items,first_purchase_date,last_purchase_date,avg_order_value
0,12346,2010-03,1,27.05,5,2010-03-02 13:08:00,2010-03-02 13:08:00,27.05
1,12346,2010-06,1,142.31,19,2010-06-28 13:53:00,2010-06-28 13:53:00,142.31
2,12346,2011-01,1,149.76,1,2011-01-18 10:01:00,2011-01-18 10:01:00,149.76
3,12347,2010-10,1,611.53,40,2010-10-31 14:20:00,2010-10-31 14:20:00,611.53
4,12347,2010-12,1,711.79,31,2010-12-07 14:57:00,2010-12-07 14:57:00,711.79
